# Condition datasets — verification

Per condition, answer two questions:

1. **Is the intended impairment present at the intended magnitude?** Every component of
   θ is re-estimated *from the data* and compared against the θ the file declares in its
   metadata — so the check is independent of the code that generated it.
2. **Did anything else change?** Shape, dtype, finiteness, labels, frame order, and — for
   the baseline — bit-for-bit equality with the clean subset.

Ordering note: nothing here is normalized. `make_subset.py` copies `X` raw and `src/data.py`
applies `unit_power` at **load** time, so injection happens strictly before normalization,
which is the physical order (hardware chain first).

In [ ]:
import sys, pathlib
# Find the repo root (the folder containing `src/`) and put it FIRST on the import path,
# so this works whether the notebook is launched from notebooks/ or the repo root.
ROOT = pathlib.Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# Drop any stale/namespace `scripts`/`src` cached by an earlier failed import.
for _m in [m for m in sys.modules if m == 'scripts' or m.startswith('scripts.')
           or m == 'src' or m.startswith('src.')]:
    del sys.modules[_m]

assert (ROOT / 'scripts' / 'check_condition.py').exists(), f"not the repo root: {ROOT}"
print('repo root:', ROOT)

import json
import numpy as np
import matplotlib.pyplot as plt

CONFIG = 'baseline_100'   # its data.path is the clean subset
SNR = 30                  # estimator variance falls with channel noise, so check high SNR

## The conditions and their θ

Defined in `configs/conditions.yaml`. Operator lists read **along the signal chain**
(phase noise first, ADC last); the thesis notation `Q_b ∘ B_d ∘ G_{a,ψ} ∘ P_σ` reads the
other way. Same chain, opposite writing direction.

In [ ]:
from scripts.make_condition import load_conditions
from src.distortions import build_compose

table = load_conditions()
for name, specs in table.items():
    chain = build_compose(specs)
    print(f"{name:14s} identity={str(chain.is_identity):5s}  {json.dumps(chain.params())}")

## Generate

One frozen file per condition, generated once and reused across every training seed.
Each is roughly the size of the clean subset (~2 GB), so this is off by default.
Equivalent CLI: `python scripts/make_condition.py --condition phase_noise --config baseline_100`

In [ ]:
from scripts.make_condition import make_condition

GENERATE = False              # flip to True to (re)generate
CONDITIONS = ['baseline', 'phase_noise', 'iq_imbalance', 'quantization', 'all']

if GENERATE:
    for condition in CONDITIONS:
        make_condition(condition, config=CONFIG, overwrite=False)
else:
    print('skipped; set GENERATE = True to write the datasets')

In [ ]:
from scripts.check_condition import check_condition

report = check_condition('phase_noise', config=CONFIG, snr=SNR, frame=0)
plt.show()

In [ ]:
for condition in ['iq_imbalance', 'quantization', 'all']:
    check_condition(condition, config=CONFIG, snr=SNR, frame=0)
    plt.show()
    print()

## Cross-condition summary

`APPROX` marks the composite chain: each estimator inverts a single operator, so with
several active at once they bias each other. Those rows are informative, not assertions —
the single-impairment conditions are where recovery is exact.

In [ ]:
rows = []
for condition in CONDITIONS:
    r = check_condition(condition, config=CONFIG, snr=SNR, plot=False, verbose=False)
    for guard, ok in r['guards']:
        rows.append((condition, guard, '', '', 'PASS' if ok else 'FAIL'))
    for p in r['recovered']:
        rows.append((condition, p['parameter'], f"{p['declared']:.4g}",
                     f"{p['recovered']:.4g}", p['status']))

print(f"{'condition':<14}{'check':<32}{'declared':>10}{'recovered':>12}   status")
for condition, name, declared, recovered, status in rows:
    print(f"{condition:<14}{name:<32}{declared:>10}{recovered:>12}   {status}")

## Open decision — the AGC reference level

Quantization and DC offset are **not** scale-invariant, so `b` bits and offset `d` only mean
something against a full-scale level. That level is an injectable `ReferenceLevel`
(`peak` / `percentile` / `fixed`) and **the choice is not yet made** — see the TODO in
`configs/conditions.yaml`.

### Why `fixed` is rejected — and not for the reason it first appears

**Clipping is not the argument.** `FS = 3.5` clips catastrophically (100% of samples in the
loudest 1% of frames), but that shows 3.5 is badly chosen, not that a fixed scale is wrong.
Branch amplitude runs ~1.0 to ~22 across this subset, so setting `FS = 26` clips *nothing*.
The cost simply reappears at the other end: the median frame then exercises ~22 of 256 codes,
so a nominal 8-bit ADC behaves like ~4.5 bits for a typical frame. Recalibrating slides along
that trade-off; it does not remove it.

**The actual reason.** RadioML's ~29 dB inter-frame amplitude spread is an artefact of how the
dataset was *synthesized* — SNR is a separate parameter there — not a property of what arrives
at an antenna. A fixed scale would carry that synthesis artefact into the ADC model disguised
as physics. Per-frame referencing leaves only a dependence on frame *shape*.

**On class dependence.** No class-neutral choice exists, and none is wanted. Under peak
referencing the quantization step scales with the frame peak, so after unit-power
normalization the effective quantization SNR is set by PAPR — which is class-dependent
(3.1–6.8 dB here). That dependence is physically correct and the dataset genuinely preserves
it. The goal is to avoid adding a *spurious* class dependence on top of it, not to remove it.

In [ ]:
import h5py
from src.config import resolve_data_path
from src.data import KEY_X, KEY_Y, MODULATION_CLASSES
from src.distortions import FixedReference, PeakReference, PercentileReference, Quantize, to_complex

N_BITS = 8
clean_path = resolve_data_path(CONFIG)[0]
with h5py.File(clean_path, 'r') as f:
    n = f[KEY_X].shape[0]
    pick = np.sort(np.random.default_rng(0).choice(n, size=1500, replace=False)).tolist()
    frames, classes = f[KEY_X][pick], f[KEY_Y][pick].argmax(1)

peaks = np.abs(frames).max(axis=(1, 2))
print(f"branch peak across frames: min {peaks.min():.2f}  median {np.median(peaks):.2f}  "
      f"max {peaks.max():.2f}\n")

# Both fixed scales are shown on purpose: recalibrating from 3.5 to 26 removes the clipping
# entirely and converts it into starvation, which is the point the table has to make.
strategies = {'peak (per-frame)': PeakReference(),
              'percentile 99.9': PercentileReference(99.9),
              'fixed FS=3.5': FixedReference(3.5),
              'fixed FS=26': FixedReference(26.0)}

header = f"{'strategy':<19}{'levels: median':>15}{'p1':>6}{'clip% p99':>11}{'loud/quiet':>12}"
print(header)
for label, reference in strategies.items():
    op = Quantize(N_BITS, reference)
    used, clipped, per_class = [], [], {}
    for frame, c in zip(frames, classes):
        x = to_complex(frame)
        fs = reference(x)
        y = op(x, np.random.default_rng(0))
        levels = np.unique(np.concatenate((y.real, y.imag))).size
        used.append(levels)
        clipped.append(np.mean(np.abs(np.concatenate((x.real, x.imag))) > fs) * 100)
        per_class.setdefault(int(c), []).append(levels)
    quiet = min(np.median(v) for v in per_class.values())
    loud = max(np.median(v) for v in per_class.values())
    print(f"{label:<19}{np.median(used):>15.0f}{np.percentile(used, 1):>6.0f}"
          f"{np.percentile(clipped, 99):>11.2f}{loud / max(quiet, 1):>11.1f}x")

print(f"\nideal is {2 ** N_BITS} levels used and 0% clipped.")
print("FS=3.5 clips; FS=26 clips nothing and starves instead -- the trade-off moves, it does")
print("not vanish. The reason to reject fixed is in the markdown above, not in this table.")